# Holdout prediction scores: IND vs BYM vs BYM+

This notebook evaluates rolling one-step-ahead weekly snow-presence probabilities over the 14-year holdout period ending in 2024 for all 1,618 cells. The training period runs from August 1972 through July 2010, and the holdout period runs from August 2010 through July 2024. The four retained metrics are probability MSE and log loss (lower is better), plus ROC-AUC and accuracy at a 0.5 threshold (higher is better).


In [ ]:
from pathlib import Path
import gc, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyreadr
from IPython.display import display
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score
from sklearn.calibration import calibration_curve
from joblib import Parallel, delayed

REPO = Path(r'path/to/Snow-Trend')
DATA_DIR = REPO / 'data'
PRED_DIR = Path(r'path/to/snow/results/holdout_38_14')
OUTPUT_DIR = REPO / 'Application' / 'prediction_evaluation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
N_TRAIN_YEARS, N_TEST_YEARS, WEEKS_PER_YEAR = 38, 14, 52
TRAIN_WEEKS = N_TRAIN_YEARS * WEEKS_PER_YEAR
TEST_WEEKS = N_TEST_YEARS * WEEKS_PER_YEAR
CHAINS = range(10)
MODEL_FILES = {
    'IND': 'pred_ind_train38_test14_chain{chain}.npy',
    'BYM': 'pred_weekly_bym_train38_test14_chain{chain}.npy',
    'BYM+': 'pred_weekly_bym_lon_train38_test14_chain{chain}.npy',
}
COLORS = {'IND': '#4477AA', 'BYM': '#EE6677', 'BYM+': '#228833'}
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})


In [ ]:
snow_obj = pyreadr.read_r(str(DATA_DIR / 'snow_cleaned_full.Rda'))
snow = next(iter(snow_obj.values())).reset_index(drop=True)
y_full = snow.iloc[:, 2:].to_numpy(dtype=np.int8)
y_test = y_full[:, TRAIN_WEEKS:TRAIN_WEEKS + TEST_WEEKS]
assert y_test.shape == (1618, 728), y_test.shape
expected_shape = (334, y_test.shape[0], y_test.shape[1])
for model, pattern in MODEL_FILES.items():
    for chain in CHAINS:
        path = PRED_DIR / pattern.format(chain=chain)
        if not path.exists():
            raise FileNotFoundError(path)
        arr = np.load(path, mmap_mode='r')
        if arr.shape != expected_shape:
            raise ValueError(f'{path.name}: {arr.shape} != {expected_shape}')
print(f'Validated 30 prediction files; each has shape {expected_shape}.')


In [ ]:
cache_path = OUTPUT_DIR / 'posterior_mean_probabilities_all_models.npz'
posterior_mean = {}
if cache_path.exists():
    cache = np.load(cache_path)
    posterior_mean = {
        'IND': cache['IND'].astype(np.float64),
        'BYM': cache['BYM'].astype(np.float64),
        'BYM+': cache['BYMplus'].astype(np.float64),
    }
    print('Loaded cached posterior means:', cache_path)
else:
    for model, pattern in MODEL_FILES.items():
        started = time.time()
        mean_sum = np.zeros(y_test.shape, dtype=np.float64)
        for chain in CHAINS:
            arr = np.load(PRED_DIR / pattern.format(chain=chain), mmap_mode='r')
            mean_sum += np.asarray(arr.mean(axis=0), dtype=np.float64)
            del arr
            gc.collect()
        posterior_mean[model] = mean_sum / 10
        print(f'{model} complete in {(time.time()-started)/60:.1f} min')
    np.savez_compressed(cache_path, IND=posterior_mean['IND'].astype(np.float32), BYM=posterior_mean['BYM'].astype(np.float32), BYMplus=posterior_mean['BYM+'].astype(np.float32))
for model, p in posterior_mean.items():
    assert p.shape == y_test.shape and np.isfinite(p).all()


In [ ]:
def binary_log_loss(y, p):
    p = np.clip(p, 1e-12, 1-1e-12)
    return -np.mean(y*np.log(p) + (1-y)*np.log1p(-p))

def prediction_scores(y, p):
    y_flat = y.ravel()
    p_flat = p.ravel()
    return {
        'Weekly probability MSE': np.mean((p_flat-y_flat)**2),
        'Log loss': binary_log_loss(y_flat, p_flat),
        'ROC-AUC': roc_auc_score(y_flat, p_flat),
        'Accuracy @ 0.5': accuracy_score(y_flat, p_flat >= 0.5),
    }

pooled_rows = [
    {'Model': model, **prediction_scores(y_test, p)}
    for model, p in posterior_mean.items()
]
scores = pd.DataFrame(pooled_rows).set_index('Model')

def score_chain(model, pattern, chain):
    arr = np.load(PRED_DIR / pattern.format(chain=chain), mmap_mode='r')
    rows = []
    for draw in range(arr.shape[0]):
        rows.append({
            'Model': model,
            'Chain': chain,
            'Draw': draw,
            **prediction_scores(y_test, np.asarray(arr[draw], dtype=np.float64)),
        })
    return pd.DataFrame(rows)

jobs = [
    (model, pattern, chain)
    for model, pattern in MODEL_FILES.items()
    for chain in CHAINS
]
draw_score_path = OUTPUT_DIR / 'prediction_scores_by_mcmc_draw.csv'
if draw_score_path.exists():
    draw_scores = pd.read_csv(draw_score_path)
    expected_rows = len(MODEL_FILES) * len(CHAINS) * expected_shape[0]
    if len(draw_scores) != expected_rows:
        raise ValueError(f'Cached draw scores have {len(draw_scores)} rows; expected {expected_rows}.')
    print(f'Loaded {len(draw_scores):,} cached draw-level score vectors.')
else:
    started = time.time()
    draw_scores = pd.concat(
        Parallel(n_jobs=3, backend='loky', verbose=10)(
            delayed(score_chain)(model, pattern, chain)
            for model, pattern, chain in jobs
        ),
        ignore_index=True,
    )
    print(f'Computed {len(draw_scores):,} draw-level score vectors in {(time.time()-started)/60:.1f} min.')

posterior_sd = draw_scores.groupby('Model')[scores.columns].std(ddof=1).loc[scores.index]
summary_columns = {}
for metric in scores.columns:
    summary_columns[metric] = scores[metric]
    summary_columns[f'{metric} posterior SD'] = posterior_sd[metric]
score_summary = pd.DataFrame(summary_columns, index=scores.index)
display(score_summary.style.format('{:.6f}'))
score_summary.to_csv(OUTPUT_DIR / 'prediction_scores_three_models.csv')
draw_scores.to_csv(draw_score_path, index=False)


In [ ]:
import subprocess
from IPython.display import Image

rscript = Path(r'Rscript')
plot_script = OUTPUT_DIR / 'plot_holdout_scores.R'
plot_path = OUTPUT_DIR / 'holdout_scores.png'
subprocess.run(
    [str(rscript), str(plot_script), str(draw_score_path), str(plot_path)],
    check=True,
)
display(Image(filename=str(plot_path)))
